In [1]:
import os
import pandas as pd
path_prefix="C://Users/nb/Desktop/paper_one/"
counties_list=os.listdir(f"{path_prefix}/kreise_daily_tas_full/")

In [3]:
from make_periodly_anomalies import make_periodly_anomalies

In [3]:
def compress_periodly_to_annual(period_df,period_flag):
    period_shorthand={'month':'mn','season':'sn'}[period_flag]
    is_compress=period_df[['Kreise_Code','tas_is_high_f1','tas_is_high_f2','tas_is_low_f1','tas_is_low_f2',
                    'prec_is_high_f1','prec_is_high_f2','prec_is_low_f1','prec_is_low_f2']].copy().reset_index()
    is_compress=is_compress.drop(period_flag,axis=1).groupby(['Kreise_Code','year']).sum()
    is_compress.columns=[f'an_{period_shorthand}_tas_is_high_f1',f'an_{period_shorthand}_tas_is_high_f2',
                         f'an_{period_shorthand}_tas_is_low_f1',f'an_{period_shorthand}_tas_is_low_f2',
                         f'an_{period_shorthand}_prec_is_high_f1',f'an_{period_shorthand}_prec_is_high_f2',
                         f'an_{period_shorthand}_prec_is_low_f1',f'an_{period_shorthand}_prec_is_low_f2']
    how_compress=[]
    for how_col in ['tas_how_high_f1','tas_how_high_f2','tas_how_low_f1','tas_how_low_f2',
                     'prec_how_high_f1','prec_how_high_f2','prec_how_low_f1','prec_how_low_f2']:
        temp_df=period_df[period_df[how_col].isna()==False][['Kreise_Code',how_col]].reset_index()
        temp_df=temp_df.drop(period_flag,axis=1).groupby(['Kreise_Code','year']).mean()
        temp_df.columns=[f'an_{period_shorthand}_{how_col}']
        how_compress.append(temp_df)
    how_compress=pd.concat(how_compress,axis=1)
    return [is_compress,how_compress]

In [3]:
sn=make_periodly_anomalies(period_flag='season',variables_flag=['prec','tas'],counties_list=counties_list)
sn.make_the_lists()

In [135]:
season=pd.concat(sn.periodly_counties,axis=0)
annual=pd.concat(sn.annually_counties,axis=0)

In [136]:
season.to_csv(f'{path_prefix}/tas_prec_seasonal_extremes.csv')

In [137]:
season=pd.read_csv(f'{path_prefix}/tas_prec_seasonal_extremes.csv')
isdf,howdf=compress_periodly_to_annual(season,period_flag='season')
final_annual=pd.concat([annual.reset_index().set_index(['Kreise_Code','year']),
                        isdf,howdf],axis=1)

In [14]:
season=pd.read_csv(f'{path_prefix}/tas_prec_seasonal_extremes.csv')
how_cols=[i for i in season.columns if 'how' in i]
season_how=season[how_cols+['Kreise_Code','year']].fillna(0)
season_how=season_how.groupby(['Kreise_Code','year']).mean()
rename_cols={}
for col in how_cols:
    new_col=f'sn0_{col}'
    rename_cols[col]=new_col
season_how=season_how.rename(columns=rename_cols)
#season_how

In [138]:
final_annual.to_csv(f'{path_prefix}/tas_prec_annual_sn_extremes.csv')

In [5]:
season_exmean=make_periodly_anomalies(period_flag='season',variables_flag=['prec','tas'],counties_list=counties_list)
season_exmean.make_extail_means()

In [16]:
season_exmeansdf=pd.concat(season_exmean.temp_c,axis=0).reset_index().set_index(['Kreise_Code','year','season'])

In [2]:
season=pd.read_csv(f'{path_prefix}/tas_prec_seasonal_extremes.csv')
#season=season.set_index(['Kreise_Code','year','season'])

In [24]:
seasonout=pd.concat([season,season_exmeansdf],axis=1)

In [23]:
seasonout.to_csv(f'{path_prefix}/tas_prec_seasonly_extremesvAug.csv')

In [33]:
mn=make_periodly_anomalies(period_flag='month',variables_flag=['prec','tas'],counties_list=counties_list)
mn.make_the_lists()

In [139]:
month=pd.concat(mn.periodly_counties,axis=0)
annual=pd.concat(mn.annually_counties,axis=0)

In [140]:
month.to_csv(f'{path_prefix}/tas_prec_monthly_extremesvAug.csv')

In [141]:
isdf,howdf=compress_periodly_to_annual(month,period_flag='month')
final_annual=pd.concat([annual.reset_index().set_index(['Kreise_Code','year']),
                        isdf,howdf],axis=1)
final_annual.to_csv(f'{path_prefix}/tas_prec_annual_mn_extremes.csv')

In [15]:
month=pd.read_csv(f'{path_prefix}/tas_prec_monthly_extremesvAug.csv')
how_cols=[i for i in month.columns if 'how' in i]
month_how=month[how_cols+['Kreise_Code','year']].fillna(0)
month_how=month_how.groupby(['Kreise_Code','year']).mean()
rename_cols={}
for col in how_cols:
    new_col=f'mn0_{col}'
    rename_cols[col]=new_col
month_how=month_how.rename(columns=rename_cols)

In [18]:
df0_hows=pd.concat([month_how,season_how],axis=1)

In [3]:
sn_stdevs=make_periodly_anomalies(period_flag='season',variables_flag=['prec','tas'],counties_list=counties_list)
sn_stdevs.make_standard_deviation()
sn_stdevs_list=sn_stdevs.periodly_devs
sn_stdevs_out=[]
for temp_df in sn_stdevs_list:
    temp_df=temp_df.reset_index()
    temp_df=temp_df[(temp_df['year']>=1995)&(temp_df['year']<=2019)]
    temp_df=temp_df.set_index(['Kreise_Code','year','season'])
    sn_stdevs_out.append(temp_df)

In [11]:
season_stdevs_df=pd.concat(sn_stdevs_out,axis=0)

In [21]:
season=pd.read_csv(f'{path_prefix}/tas_prec_seasonal_extremes.csv')
season=season.set_index(['Kreise_Code','year','season'])

In [22]:
season_out=pd.concat([season,season_stdevs_df],axis=1)
season_out.to_csv(f'{path_prefix}/tas_prec_seasonal_extremesvAug.csv')

In [12]:
mn_stdevs=make_periodly_anomalies(period_flag='month',variables_flag=['prec','tas'],counties_list=counties_list)
mn_stdevs.make_standard_deviation()
mn_stdevs_list=mn_stdevs.periodly_devs
mn_stdevs_out=[]
for temp_df in mn_stdevs_list:
    temp_df=temp_df.reset_index()
    temp_df=temp_df[(temp_df['year']>=1995)&(temp_df['year']<=2019)]
    temp_df=temp_df.set_index(['Kreise_Code','year','month'])
    mn_stdevs_out.append(temp_df)

In [13]:
month_stdevs_df=pd.concat(mn_stdevs_out,axis=0)

In [10]:
month=pd.read_csv(f'{path_prefix}/tas_prec_monthly_extremesvAug.csv')
month=month.set_index(['Kreise_Code','year','month'])

In [17]:
month_out=pd.concat([month,month_stdevs_df],axis=1)
month_out.to_csv(f'{path_prefix}/tas_prec_monthly_extremesvAug.csv')

In [6]:
month_exmean=make_periodly_anomalies(period_flag='month',variables_flag=['prec','tas'],counties_list=counties_list)
month_exmean.make_extail_means()

In [9]:
month_exmeansdf=pd.concat(month_exmean.temp_c,axis=0).reset_index().set_index(['Kreise_Code','year','month'])

In [15]:
monthout=pd.concat([month,month_exmeansdf],axis=1)
monthout.to_csv(f'{path_prefix}/tas_prec_monthly_extremesvAug.csv')

In [27]:
temp_month=monthout[['month_tas_mean_f1','month_tas_mean_f2','month_prec_mean_f1','month_prec_mean_f2']].reset_index().drop('month',axis=1)
temp_season=seasonout[['season_tas_mean_f1','season_tas_mean_f2','season_prec_mean_f1','season_prec_mean_f2']].reset_index().drop('season',axis=1)
temp_month=temp_month.groupby(['Kreise_Code','year']).std()
temp_month.columns=['exmn_tas_stdev_f1','exmn_tas_stdev_f2','exmn_prec_stdev_f1','exmn_prec_stdev_f2']
temp_season=temp_season.groupby(['Kreise_Code','year']).std()
temp_season.columns=['exsn_tas_stdev_f1','exsn_tas_stdev_f2','exsn_prec_stdev_f1','exsn_prec_stdev_f2']

In [142]:
gdppc_filename="/gdppc_sn_klima_may2026vAug.csv"
givendf=pd.read_csv(path_prefix+gdppc_filename)
givendf=givendf[['Kreise_Code','EU_Code','year','ln_gdppc','ln_gdppc.d1']]

In [143]:
sn_annual=pd.read_csv(f'{path_prefix}/tas_prec_annual_sn_extremes.csv')
outdf_sn=pd.merge(sn_annual,givendf,on=['Kreise_Code','year'])
outdf_sn.to_csv(path_prefix+"/gdppc_sn_klima_may2026vAug2.csv",index=False)

In [144]:
mn_annual=pd.read_csv(f'{path_prefix}/tas_prec_annual_mn_extremes.csv')
outdf_mn=pd.merge(mn_annual,givendf,on=['Kreise_Code','year'])
outdf_mn.to_csv(path_prefix+"/gdppc_mn_klima_may2026vAug2.csv",index=False)

In [153]:
outdf_sn=outdf_sn.drop(['ln_gdppc','ln_gdppc.d1'],axis=1)
outdf=pd.merge(outdf_sn,outdf_mn,on=['Kreise_Code','EU_Code','year'])

In [155]:
outdf.to_csv(path_prefix+"/gdppc_snmn_klima_may2026vAug2.csv",index=False)

In [19]:
outdf=pd.read_csv(path_prefix+"/gdppc_snmn_klima_may2026vAug2.csv")
outdf=outdf.set_index(['Kreise_Code','year'])

In [35]:
outdf=outdf.fillna(0)
outdf0=pd.concat([outdf,df0_hows],axis=1)
#test=outdf0[[i for i in outdf0.columns if 'how' in i]].dropna()
outdf0=outdf0.dropna()

In [36]:
outdf0.to_csv(path_prefix+"/gdppc_snmn_klima_may2026vAug3.csv")

In [30]:
outdf=pd.read_csv(path_prefix+"/gdppc_snmn_klima_may2026vAug3.csv").set_index(['Kreise_Code','year'])
outdf

sn_mean_prec  sn_stdev_prec_type1  sn_stdev_prec_type2  \
Kreise_Code year                                                           
1001        1995     15.765763             8.093076            14.500264   
            1996     10.491841             6.284781            11.371561   
            1997     14.934814             2.774178            16.922176   
            1998     22.466091             3.746691            19.061036   
            1999     20.776697             6.797085            18.093750   
...                        ...                  ...                  ...   
9780        2015     25.980134             8.452333            23.189938   
            2016     30.595122            11.086955            21.744622   
            2017     34.511656             9.130151            25.790837   
            2018     24.185799             2.399651            24.449990   
            2019     31.787738             3.537255            27.386444   

                  sn_mean_tas  sn_stdev_tas_type1  sn_stdev_tas_type2  \
Kreise_Code year                                                        
1001        1995     8.774718            6.290887            4.286743   
            1996     7.170579            7.102072            3.777245   
            1997     8.981304            6.685594            3.980986   
            1998     8.761699            5.207577            3.441961   
            1999     9.393812            6.201513            3.308682   
...                       ...                 ...                 ...   
9780        2015     7.770816            6.857624            4.550764   
            2016     7.420746            6.665036            3.973104   
            2017     7.230938            6.635674            5.465519   
            2018     8.133547            7.563323            4.186602   
            2019     7.603471            6.632377            4.619243   

                  an_sn_tas_is_high_f1  an_sn_tas_is_high_f2  \
Kreise_Code year                                               
1001        1995                  49.0                  45.0   
            1996                  17.0                  13.0   
            1997                  56.0                  44.0   
            1998                  30.0                  26.0   
            1999                  29.0                  20.0   
...                                ...                   ...   
9780        2015                  49.0                  33.0   
            2016                  25.0                  15.0   
            2017                  58.0                  43.0   
            2018                  36.0                  23.0   
            2019                  42.0                  28.0   

                  an_sn_tas_is_low_f1  an_sn_tas_is_low_f2  ...  \
Kreise_Code year                                            ...   
1001        1995                 16.0                 14.0  ...   
            1996                 20.0                 19.0  ...   
            1997                  8.0                  8.0  ...   
            1998                 10.0                 10.0  ...   
            1999                  2.0                  2.0  ...   
...                               ...                  ...  ...   
9780        2015                  8.0                 15.0  ...   
            2016                  3.0                  5.0  ...   
            2017                 14.0                 20.0  ...   
            2018                  7.0                  8.0  ...   
            2019                  1.0                  5.0  ...   

                  mn0_tas_how_high_f2  mn0_tas_how_low_f2  \
Kreise_Code year                                            
1001        1995             0.859049            0.259272   
            1996             0.792501            0.448923   
            1997             0.545277            0.180560   
            1998             0.708701            0.352483   
        

In [32]:
outtemp=pd.concat([outdf,temp_month,temp_season],axis=1).reset_index()
outtemp=outtemp[(outtemp['year']>=1995)&(outtemp['year']<=2019)]
outtemp

,Kreise_Code,year,sn_mean_prec,sn_stdev_prec_type1,sn_stdev_prec_type2,sn_mean_tas,sn_stdev_tas_type1,sn_stdev_tas_type2,an_sn_tas_is_high_f1,an_sn_tas_is_high_f2,...,sn0_tas_how_high_f2,sn0_tas_how_low_f2,exmn_tas_stdev_f1,exmn_tas_stdev_f2,exmn_prec_stdev_f1,exmn_prec_stdev_f2,exsn_tas_stdev_f1,exsn_tas_stdev_f2,exsn_prec_stdev_f1,exsn_prec_stdev_f2
0,1001,1995,15.765763,8.093076,14.500264,8.774718,6.290887,4.286743,49.0,45.0,...,1.608955,1.604732,6.215575,6.267760,5.607074,5.510599,6.105769,6.198351,3.473439,3.473439
1,1001,1996,10.491841,6.284781,11.371561,7.170579,7.102072,3.777245,17.0,13.0,...,1.129793,1.438234,6.808309,6.827072,7.951657,7.382749,6.708525,6.777447,6.461876,4.864809
2,1001,1997,14.934814,2.774178,16.922176,8.981304,6.685594,3.980986,56.0,44.0,...,1.290758,0.864281,6.340003,6.446169,7.371787,7.301108,6.274934,6.214147,2.092653,2.092653
3,1001,1998,22.466091,3.746691,19.061036,8.761699,5.207577,3.441961,30.0,26.0,...,1.078541,0.979232,5.596854,5.485397,6.356793,7.453179,5.423241,5.384885,3.473196,3.917195
4,1001,1999,20.776697,6.797085,18.093750,9.393812,6.201513,3.308682,29.0,20.0,...,0.964531,0.192844,5.896459,5.880722,7.744214,7.398174,6.144739,6.169752,5.286974,4.210661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9780,2015,25.980134,8.452333,23.189938,7.770816,6.857624,4.550764,49.0,33.0,...,1.393192,0.661159,6.508549,6.644815,9.355345,9.568789,6.205833,6.551306,8.058907,8.058907
9996,9780,2016,30.595122,11.086955,21.744622,7.420746,6.665036,3.973104,25.0,15.0,...,1.587687,1.460916,6.496641,6.458317,14.835551,14.534993,6.562616,6.650943,10.503305,10.503305
9997,9780,2017,34.511656,9.130151,25.790837,7.230938,6.635674,5.465519,58.0,43.0,...,1.301630,1.233367,6.913262,7.080996,5.340933,6.783158,6.399527,6.503647,3.737981,3.886668
9998,9780,2018,24.185799,2.399651,24.449990,8.133547,7.563323,4.186602,36.0,23.0,...,0.652380,1.940191,7.222836,7.159654,10.266611,11.007830,7.130644,7.257725,3.471230,3.618262


In [33]:
outtemp.to_csv(path_prefix+"/gdppc_snmn_klima_may2026vAug3.csv",index=False)

In [34]:
[i for i in outtemp.columns if 'stdev' in i]

['sn_stdev_prec_type1',
 'sn_stdev_prec_type2',
 'sn_stdev_tas_type1',
 'sn_stdev_tas_type2',
 'mn_stdev_prec_type1',
 'mn_stdev_prec_type2',
 'mn_stdev_tas_type1',
 'mn_stdev_tas_type2',
 'exmn_tas_stdev_f1',
 'exmn_tas_stdev_f2',
 'exmn_prec_stdev_f1',
 'exmn_prec_stdev_f2',
 'exsn_tas_stdev_f1',
 'exsn_tas_stdev_f2',
 'exsn_prec_stdev_f1',
 'exsn_prec_stdev_f2']

In [35]:
outtemp[['exmn_tas_stdev_f1','exmn_tas_stdev_f2','mn_stdev_tas_type1']].describe()

,exmn_tas_stdev_f1,exmn_tas_stdev_f2,mn_stdev_tas_type1
count,10000.000000,10000.000000,10000.000000
mean,6.623265,6.604817,6.772523
std,0.624644,0.630151,0.762547
min,5.002054,4.895037,4.730266
25%,6.189766,6.175896,6.247638
50%,6.635584,6.632562,6.798453
75%,7.057323,7.045381,7.293114
max,8.708794,8.619223,9.130168


#### Q. Why do "mn0_how_high_mean_f1" and "mn0_how_high_mean_f2" have much stronger effect than seasonal measures, and especially when sn0 measures are statistically insignificant?

In [15]:
season=pd.read_csv(f'{path_prefix}/tas_prec_seasonal_extremes.csv')
month=pd.read_csv(f'{path_prefix}/tas_prec_monthly_extremesvAug.csv')
season=season[['Kreise_Code','year','season','tas_how_high_f1','tas_how_high_f2']]
month=month[['Kreise_Code','year','month','tas_uqtl_f1','tas_how_high_f1','tas_uqtl_f2','tas_how_high_f2']]
dresden=month[((month['year']>=1995)&(month['year']<=1997))&(month['Kreise_Code']==14612)]
berlin=month[((month['year']>=1995)&(month['year']<=1997))&(month['Kreise_Code']==11000)]
# dresden[dresden['tas_uqtl_f1']>dresden['tas_uqtl_f2']]
# berlin[berlin['tas_uqtl_f1']>berlin['tas_uqtl_f2']]

,Kreise_Code,year,month,tas_uqtl_f1,tas_how_high_f1,tas_uqtl_f2,tas_how_high_f2
19777,14612,1995,2,6.935043,0.980271,6.855652,1.059662
19779,14612,1995,4,15.348591,3.693148,14.422022,4.619718
19781,14612,1995,6,22.515805,NaN,22.488444,NaN
19784,14612,1995,9,19.463063,NaN,19.440534,NaN
19789,14612,1996,2,6.935043,NaN,6.795887,NaN
19791,14612,1996,4,15.348591,3.480008,14.422022,3.611426
19793,14612,1996,6,22.515805,1.266210,22.488444,1.293570
19796,14612,1996,9,19.463063,NaN,19.449606,NaN
19801,14612,1997,2,6.935043,1.514164,6.855652,1.593555
19803,14612,1997,4,15.348591,NaN,14.185371,NaN


,Kreise_Code,year,month,tas_uqtl_f1,tas_how_high_f1,tas_uqtl_f2,tas_how_high_f2
7875,11000,1995,4,15.518601,3.590827,15.081005,4.028422
7880,11000,1995,9,19.450752,0.372936,19.353185,0.470503
7887,11000,1996,4,15.518601,3.555208,14.957587,4.116222
7892,11000,1996,9,19.450752,NaN,19.440140,NaN
7899,11000,1997,4,15.518601,NaN,14.662151,NaN
7904,11000,1997,9,19.450752,1.619878,19.302936,1.767694
